# Telecom Customer Churn Analysis and AI-Based Churn Prediction

**IBM SkillsBuild Data Analytics with AI Internship – Final Project**

### Project objective
Analyze customer behavior in a telecommunications dataset and build machine-learning models that can identify customers who are more likely to churn. The analysis combines data cleaning, exploratory data analysis (EDA), visualization, feature engineering, classification models, evaluation, and business recommendations.

**Dataset:** Telco Customer Churn — IBM Sample Data Sets, available through Kaggle.  
**Kaggle:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn

> **Important:** Download `WA_Fn-UseC_-Telco-Customer-Churn.csv` from Kaggle and place it in the same folder as this notebook before running it.


## 1. Business Problem

Customer churn can reduce recurring revenue and increase the cost of acquiring replacement customers. The goal of this project is to answer:

1. What customer characteristics and services are associated with churn?
2. Which customer groups show higher churn rates?
3. Can machine-learning models predict whether a customer will churn?
4. Which model provides useful predictive performance?
5. What practical retention actions can be derived from the analysis?

### Project scope
This is an educational analytics project using a cross-sectional sample dataset. Model predictions should be treated as analytical estimates, not as proof that a particular retention action will cause a customer to stay.


In [ ]:
# 2. Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
# 3. Load the dataset
DATA_FILE = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

try:
    df = pd.read_csv(DATA_FILE)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Download it from Kaggle and place it next to this notebook."
    )

print(f"Dataset shape: {df.shape}")
display(df.head())


## 4. Initial Data Inspection

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.info())
display(df.describe(include="all").T)


In [ ]:
# Missing values and duplicate IDs
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values:")
display(missing[missing > 0].to_frame("missing_count"))

print("Duplicate customer IDs:", df["customerID"].duplicated().sum())


## 5. Data Cleaning

`TotalCharges` is stored as text in the Kaggle file and may contain blank strings for some customers. We convert it to numeric values and handle the resulting missing values.

`customerID` is an identifier rather than a useful predictive feature, so it is removed before modeling.


In [ ]:
# Clean TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges after conversion:", df["TotalCharges"].isna().sum())

# Fill missing TotalCharges with 0 only when tenure is 0; otherwise use the median.
df.loc[df["TotalCharges"].isna() & (df["tenure"] == 0), "TotalCharges"] = 0
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Remove duplicate rows if any
df = df.drop_duplicates().copy()

print("Final shape after cleaning:", df.shape)
display(df.head())


## 6. Exploratory Data Analysis (EDA)

In [ ]:
# Churn distribution
churn_counts = df["Churn"].value_counts()
churn_rates = df["Churn"].value_counts(normalize=True).mul(100).round(2)

print("Churn counts:")
display(churn_counts)
print("Churn percentages:")
display(churn_rates)

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()


In [ ]:
# Numerical distributions
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.histplot(df["tenure"], kde=True, ax=axes[0])
axes[0].set_title("Tenure Distribution")

sns.histplot(df["MonthlyCharges"], kde=True, ax=axes[1])
axes[1].set_title("Monthly Charges Distribution")

sns.histplot(df["TotalCharges"], kde=True, ax=axes[2])
axes[2].set_title("Total Charges Distribution")

plt.tight_layout()
plt.show()


In [ ]:
# Churn rate helper
def churn_rate_by(column):
    result = (
        df.groupby(column)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .sort_values(ascending=False)
        .round(2)
        .reset_index(name="ChurnRate")
    )
    return result

for col in ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]:
    print(f"\nChurn rate by {col}:")
    display(churn_rate_by(col))


In [ ]:
# Contract type and churn
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100

contract_churn.plot(kind="bar", figsize=(9, 5))
plt.title("Churn Percentage by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Percentage of Customers")
plt.xticks(rotation=0)
plt.legend(title="Churn")
plt.tight_layout()
plt.show()


In [ ]:
# Internet service and churn
internet_churn = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100

internet_churn.plot(kind="bar", figsize=(9, 5))
plt.title("Churn Percentage by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Percentage of Customers")
plt.xticks(rotation=0)
plt.legend(title="Churn")
plt.tight_layout()
plt.show()


In [ ]:
# Payment method and churn
payment_churn = pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index") * 100

payment_churn.plot(kind="bar", figsize=(11, 5))
plt.title("Churn Percentage by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Percentage of Customers")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Churn")
plt.tight_layout()
plt.show()


In [ ]:
# Tenure vs churn
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure Distribution by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Tenure (months)")
plt.tight_layout()
plt.show()

# Monthly charges vs churn
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.tight_layout()
plt.show()


In [ ]:
# Service-related churn analysis
service_columns = [
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies"
]

service_results = []
for col in service_columns:
    tmp = churn_rate_by(col)
    tmp.insert(0, "Feature", col)
    service_results.append(tmp)

service_summary = pd.concat(service_results, ignore_index=True)
display(service_summary.sort_values("ChurnRate", ascending=False).head(20))


## 7. Correlation Analysis

For numerical variables, correlation can help identify relationships between variables. Correlation does **not** imply causation.


In [ ]:
numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()


## 8. Feature Engineering

Two interpretable features are added:

- `TenureGroup`: groups customers by length of relationship.
- `TotalServices`: counts selected subscribed services.

These are used for exploratory analysis and modeling.


In [ ]:
# Tenure groups
bins = [-1, 12, 24, 48, 72, np.inf]
labels = ["0-12", "13-24", "25-48", "49-72", "73+"]
df["TenureGroup"] = pd.cut(df["tenure"], bins=bins, labels=labels)

# Count selected services
service_yes_columns = [
    "PhoneService", "MultipleLines", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies"
]

df["TotalServices"] = sum(
    (df[col] == "Yes").astype(int)
    for col in service_yes_columns
)

display(df[["tenure", "TenureGroup", "TotalServices", "Churn"]].head())


In [ ]:
# Churn by tenure group
tenure_group_churn = churn_rate_by("TenureGroup")

plt.figure(figsize=(9, 5))
sns.barplot(data=tenure_group_churn, x="TenureGroup", y="ChurnRate")
plt.title("Churn Rate by Tenure Group")
plt.xlabel("Tenure Group (months)")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()


## 9. Machine Learning Preparation

The target variable is `Churn`.

- `Yes` → 1
- `No` → 0

Categorical variables are one-hot encoded.
Numerical variables are standardized.
The preprocessing is placed inside a scikit-learn pipeline to avoid data leakage from the test set.


In [ ]:
# Prepare modeling data
model_df = df.drop(columns=["customerID"]).copy()
model_df["Churn"] = model_df["Churn"].map({"Yes": 1, "No": 0})

X = model_df.drop(columns=["Churn"])
y = model_df["Churn"]

# Use stratification because churn classes are not equally represented.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numeric_features))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


## 10. Train Classification Models

We compare three common classification algorithms:

1. Logistic Regression
2. Decision Tree
3. Random Forest

Model comparison is based on accuracy, precision, recall, F1-score and ROC-AUC.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=3,
        class_weight="balanced", random_state=42, n_jobs=-1
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
display(results_df.round(4))


In [ ]:
# Compare ROC curves
plt.figure(figsize=(9, 6))

for name, pipeline in trained_models.items():
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=name, ax=plt.gca()
    )

plt.title("ROC Curves – Churn Prediction Models")
plt.tight_layout()
plt.show()


In [ ]:
# Detailed evaluation of the model with the highest ROC-AUC
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

best_pred = best_model.predict(X_test)

print("Selected model for detailed evaluation:", best_model_name)
print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=["No Churn", "Churn"]))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_pred,
    display_labels=["No Churn", "Churn"],
    cmap="Blues"
)
plt.title(f"Confusion Matrix – {best_model_name}")
plt.tight_layout()
plt.show()


## 11. Feature Importance / Model Interpretation

For tree-based models, feature importance provides an indication of which transformed variables contributed most to the model's decisions. It should be interpreted as model behavior, not as proof of causality.


In [ ]:
# Feature importance when the selected model is Random Forest or Decision Tree
if best_model_name in ["Random Forest", "Decision Tree"]:
    fitted_preprocessor = best_model.named_steps["preprocessor"]
    fitted_model = best_model.named_steps["model"]

    feature_names = fitted_preprocessor.get_feature_names_out()
    importances = fitted_model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False).head(20)

    display(importance_df)

    plt.figure(figsize=(10, 7))
    sns.barplot(data=importance_df, x="Importance", y="Feature")
    plt.title(f"Top Model Features – {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    print("The selected model is Logistic Regression; tree feature importance is not applicable.")


## 12. Business Insights

Based on the EDA and model results, review the following areas when interpreting customer churn:

- Contract type and customer tenure
- Monthly charges
- Internet service type
- Payment method
- Availability of support/security services
- Customer service combinations

Potential retention actions should be targeted and tested rather than assuming that a correlation is causal. The model can be used to prioritize customers for review, while the business should validate interventions with controlled experiments or follow-up analysis.


In [ ]:
# Generate a concise analytical summary from the dataset
overall_churn = (df["Churn"] == "Yes").mean() * 100
avg_monthly_churn = df.loc[df["Churn"] == "Yes", "MonthlyCharges"].mean()
avg_monthly_nonchurn = df.loc[df["Churn"] == "No", "MonthlyCharges"].mean()
avg_tenure_churn = df.loc[df["Churn"] == "Yes", "tenure"].mean()
avg_tenure_nonchurn = df.loc[df["Churn"] == "No", "tenure"].mean()

print(f"Overall churn rate: {overall_churn:.2f}%")
print(f"Average monthly charges – churned customers: {avg_monthly_churn:.2f}")
print(f"Average monthly charges – non-churned customers: {avg_monthly_nonchurn:.2f}")
print(f"Average tenure – churned customers: {avg_tenure_churn:.2f} months")
print(f"Average tenure – non-churned customers: {avg_tenure_nonchurn:.2f} months")

print("\nHighest churn-rate contract group:")
display(churn_rate_by("Contract").head(1))

print("Highest churn-rate internet service group:")
display(churn_rate_by("InternetService").head(1))

print("Highest churn-rate payment method group:")
display(churn_rate_by("PaymentMethod").head(1))


## 13. Conclusion

This project demonstrates an end-to-end data analytics and AI workflow:

- Dataset understanding
- Data quality checking and cleaning
- Exploratory data analysis
- Visualization
- Feature engineering
- Classification modeling
- Model evaluation
- Model interpretation
- Business-oriented recommendations

The final model should be evaluated using more than accuracy because churn prediction is a class-imbalanced problem. Recall, F1-score and ROC-AUC provide additional information about the model's usefulness.

### Limitations

1. The dataset is a sample dataset for educational/analytical use.
2. It is a cross-sectional customer snapshot rather than a sequence of monthly observations.
3. Predictive association does not establish causation.
4. Model performance on this dataset does not guarantee the same performance in another telecom company or future population.


## 14. Reproducibility

### Required dataset file

Download the Kaggle dataset and place:

`WA_Fn-UseC_-Telco-Customer-Churn.csv`

in the same directory as this notebook.

### Main Python libraries

- pandas
- numpy
- matplotlib
- seaborn
- scikit-learn

See `requirements.txt` in the final project repository for the exact dependency list.
